# 03 — Perfiles: la conjunta más grande posible

Parte D (D1–D4) de `microdatos.ipynb`, migrada al módulo `manzanas/`. Lee la espina de `resultados/espina_13108.npz` (notebook 02), construye el inventario de marginales, ajusta el modelo de 8 bloques (D3) y el de 27 bloques (D4), y guarda ambos en `resultados/` para el notebook 04.

**D4c tarda ~15–20 min en GPU.** Si `resultados/modelo_27_13108.npz` ya existe, lo carga en vez de reajustar; con `REAJUSTAR = True` se fuerza el ajuste.

# Parte D — La conjunta más grande posible: representación por perfiles

Con la espina validada, la pregunta es cuántas variables podemos meter a la vez. El tensor denso no escala: los 27 bloques de variables de personas con marginal por manzana (82 columnas de conteo `n_*`) dan más de $10^{12}$ celdas de atributos, multiplicadas por 469 manzanas.

Pero la conjunta comunal $\mu$ **es el microdato**: tiene a lo más $N = 116.943$ celdas no nulas. Como IPF es multiplicativo, una celda con $\mu[x]=0$ queda en cero para siempre. Toda la masa vive en los **perfiles observados** en la comuna.

**Formulación.** Sea $\mathcal{P}$ el conjunto de perfiles distintos $x_p=(x_{p1},\dots,x_{pK})$ que aparecen en `df_personas_com` para las $K$ variables con marginal en la manzana, y sea $\mu_p$ el número de personas con ese perfil. La incógnita es una matriz $W\in\mathbb{R}_{\ge0}^{|\mathcal P|\times M}$:

$$
W^* = \arg\min_{W\ge0} D_{KL}\!\left(W\,\|\,W^{(0)}\right)\quad\text{s.a.}\quad
\sum_{m} W[p,m] = \mu_p\;\;\forall p,\qquad
\sum_{p:\,x_{pk}=c} W[p,m] = \nu^{(k)}[c,m]\;\;\forall k,c,m .
$$

Es exactamente el IPF denso con $\mu$ igual a la conjunta comunal completa de las $K$ variables; `T_espina` es el caso $K=3$, $|\mathcal P|=28$ (D1 lo verifica). En Independencia, con los 27 bloques, $|\mathcal P| = 40.946$, así que $W$ tiene 19,2 millones de celdas. Con las 63 variables de personas a la vez, $|\mathcal P|\le N$ y $W$ tiene a lo más 55 millones.

**Las variables sin marginal en la manzana (nivel 2) salen sin ajuste adicional.** Cada persona $i$ del microdato comunal hereda la distribución espacial de su perfil $p(i)$:

$$
\widehat{T}[y,m] = \sum_{i:\,y_i=y} \frac{W[p(i),m]}{\mu_{p(i)}} ,
$$

para cualquier combinación $y$ de variables, tengan o no marginal publicado (país de nacimiento, año de llegada, edad simple, parentesco, escolaridad, etc.). Es la extensión cerrada del nivel 2, pero condicionando en el perfil **completo** y no solo en $(e,s,i)$, y sigue siendo la proyección KL con $\mu$ igual a la conjunta comunal de todas las variables. El supuesto es que, dado el perfil completo, la distribución de $y$ no depende de la manzana: es inferencia, no dato.

**Advertencias.**
- *Privacidad.* El 75% de los perfiles (30.742) corresponde a una sola persona. Una fila $W[p,:]$ es entonces la ubicación probable de un individuo identificable en el microdato comunal, que es justamente la asignación persona→manzana que se descartó. $W$ es un objeto interno y solo se publican tablas agregadas. En la Fase 2, con ruido de Laplace sobre $\nu$, $W$ es post-procesamiento de ($\mu$ público, $\tilde\nu$ privado) y hereda la garantía. Pero cada bloque de restricciones suma sensibilidad (una persona cae en una categoría de cada bloque): más restricciones implican más presupuesto $\varepsilon$ o más ruido por marginal.
- *Unidades.* Hogares y viviendas tienen sus propios marginales (36 y 68 columnas) y necesitan su propia $W$. Los marginales de hogar que dependen de la composición (`n_hog_unipersonales`, `n_hog_60`, `n_hog_menores`, `n_jefatura_mujer`) son el puente con personas del nivel 3.

**Plan de esta parte.** D1 verifica que `ipf_perfiles` reproduce `T_espina`. D2 construye el inventario de los 82 marginales de personas con su mapeo al diccionario, verificado contra los totales comunales. Después se agregan los bloques uno a uno: antes de agregar cada bloque se predice su marginal con el modelo vigente (LOO externo), y luego se ajusta y se verifica.

In [1]:
import os

import numpy as np
import pandas as pd

from manzanas.datos import cargar_comuna
from manzanas.inventario import EDAD_TRAMOS, construir_inventario
from manzanas.ipf import ipf_perfiles
from manzanas.perfiles import (ajustar_modelo, ajustar_modelo_gpu, cargar_modelo, codigos_bloque, construir_perfiles,
                               errores_modelo, guardar_modelo, indicadora, predecir_bloque)
from manzanas.validacion import errores_marginales, srmse

pd.set_option("display.width", 250)
COMUNA_CUT = 13108
datos = cargar_comuna(COMUNA_CUT)
nu, manzanas_ids = datos.nu, datos.manzanas_ids

esp = np.load(os.path.join("resultados", f"espina_{COMUNA_CUT}.npz"))  # notebook 02
T_espina, mu_edad_sexo_inmigrante = esp["T_espina"], esp["mu_edad_sexo_inmigrante"]
nu_edad, nu_sexo, nu_inmigrante, mask_enmascarada = esp["nu_edad"], esp["nu_sexo"], esp["nu_inmigrante"], esp["mask_enmascarada"]
constraints_espina = [((3,), mu_edad_sexo_inmigrante), ((1, 2), nu_edad), ((0, 2), nu_sexo), ((0, 1), nu_inmigrante)]
nombres_espina = ["mu(e,s,i)", "nu(e,m)", "nu(s,m)", "nu(i,m)"]
print("personas:", len(datos.personas), "| manzanas:", len(nu), "| T_espina:", T_espina.shape, "| total:", round(T_espina.sum(), 6))
assert abs(T_espina.sum() - len(datos.personas)) < 1e-6

personas: 116943 | manzanas: 469 | T_espina: (7, 2, 2, 469) | total: 116943.0


### D1. `ipf_perfiles` y regresión contra `T_espina`

`ipf_perfiles` está en `manzanas/ipf.py`. Las sumas por categoría se calculan con una matriz dispersa de indicadores $E_k$ de tamaño $C_k\times|\mathcal P|$, con $\operatorname{proj}_k(W)=E_kW$. Es la misma álgebra que `ipf` sobre el tensor denso.

In [2]:
# D1. Regresión: los perfiles (edad, sexo, inmigrante) deben reproducir T_espina
edad_np = datos.personas["edad"].to_numpy()
edad_idx = np.full(len(edad_np), -1)
for i, (_, lo, hi) in enumerate(EDAD_TRAMOS):
    edad_idx[(edad_np >= lo) & (edad_np <= hi)] = i
assert (edad_idx >= 0).all()
es_inmigrante = (datos.personas["p25_lug_nacimiento_rec"].to_numpy() == 2).astype(int)

nombres_3, perfiles_3, perfil_persona_3, mu_p_3 = construir_perfiles({
    "edad": edad_idx,
    "sexo": datos.personas["sexo"].to_numpy() - 1,  # 0 = hombre, 1 = mujer
    "inmigrante": 1 - es_inmigrante,                 # 0 = inmigrante, 1 = resto (mismo orden que nu_inmigrante)
})
print("perfiles (edad, sexo, inmigrante):", len(mu_p_3), "| personas:", mu_p_3.sum())

W_3, n_it_w, conv_w = ipf_perfiles(mu_p_3, [perfiles_3[:, 0], perfiles_3[:, 1], perfiles_3[:, 2]],
                                   [nu_edad, nu_sexo, nu_inmigrante])
print("iteraciones:", n_it_w, "| convergió:", conv_w, "| forma W:", W_3.shape, "| (C3 tomó 5242)")

T_desde_W = np.zeros_like(T_espina)
T_desde_W[perfiles_3[:, 0], perfiles_3[:, 1], perfiles_3[:, 2], :] = W_3
print("max |T_desde_W - T_espina|:", np.abs(T_desde_W - T_espina).max())
for k, v in errores_marginales(T_desde_W, constraints_espina, nombres_espina).items():
    print(f"  error marginal {k}: {v:.2e}")

assert conv_w
assert np.abs(T_desde_W - T_espina).max() < 1e-6

perfiles (edad, sexo, inmigrante): 28 | personas: 116943
iteraciones: 5242 | convergió: True | forma W: (28, 469) | (C3 tomó 5242)
max |T_desde_W - T_espina|: 0.0
  error marginal mu(e,s,i): 6.58e-07
  error marginal nu(e,m): 2.81e-08
  error marginal nu(s,m): 1.34e-09
  error marginal nu(i,m): 4.55e-13


### D2. Inventario de marginales de personas

El mapeo (`MAPEO_PERSONAS` en `manzanas/inventario.py`) se leyó de `diccionario_variables_glosas_censo2024_manzana_agregados.xlsx` (columnas $\nu$) y de `diccionario_variables_censo2024_nivel_comuna.xlsx` (códigos del microdato). Cada columna se verifica contra el total comunal: debe ser exacta si no tiene `*`, y con residuo $\ge 0$ si lo tiene.

Notas del diccionario:
- Las dificultades cuentan *mucha dificultad* (3) o *no puede* (4). `discapacidad` es función determinística de las seis (agregarla no cambia el número de perfiles), pero `n_discapacidad` sí aporta información: restringe la conjunta de las dificultades en cada manzana.
- `n_analfabet` tiene universo 15+ y `p37_alfabet` tiene universo 5+, así que se restringe por edad.
- Las asistencias coinciden con los tramos de edad (parvularia 0–5, básica 6–13, media 14–17, superior 18–24).
- `-99` (no respuesta) y `NA` (no aplica) no se publican en la manzana. Quedan en una categoría *otro* sin restricción, cuyo total por manzana queda implícito por `n_per`.
- El código `-66` (valor suprimido por anonimización) no aparece en Independencia (`comuna_bajo_umbral = 2`). En las comunas bajo el umbral el microdato tendrá `-66` y el total comunal no va a calzar con la suma de manzanas; hay que tratarlo antes de escalar a otras comunas.

In [3]:
# D2. Inventario de marginales de personas, verificado contra los totales comunales
inv = construir_inventario(datos)
print(inv.tabla.to_string(index=False))

# Consistencia con lo usado en la espina
assert np.array_equal(inv.marginales["edad"]["nu"], nu_edad)
assert np.array_equal(inv.marginales["sexo"]["nu"], nu_sexo)
assert np.array_equal(inv.marginales["inmigrante"]["nu"][0], nu_inmigrante[0], equal_nan=True)

n_m66 = {v: int((inv.per[v] == -66).sum()) for v in inv.per.columns}
print("\nbloques:", len(inv.marginales), "| columnas:", len(inv.tabla), "| con '*':", int((inv.tabla["enmascaradas"] > 0).sum()))
print("variables con -66:", {v: n for v, n in n_m66.items() if n > 0} or "ninguna",
      "| comuna_bajo_umbral:", datos.personas["comuna_bajo_umbral"].unique())

        bloque                             columna                 variable            códigos  total comuna  suma manzanas  enmascaradas  residuo
          edad                          n_edad_0_5                     edad [0, 1, 2, 3, 4, 5]          7554           7554             0      0.0
          edad                         n_edad_6_13                     edad              6..13         11542          11542             0      0.0
          edad                        n_edad_14_17                     edad   [14, 15, 16, 17]          5280           5280             0      0.0
          edad                        n_edad_18_24                     edad             18..24         11439          11439             0      0.0
          edad                        n_edad_25_44                     edad             25..44         44178          44178             0      0.0
          edad                        n_edad_45_59                     edad             45..59         19274          

### D3. Agregar bloques uno a uno: predicción previa y ajuste

Para cada bloque nuevo $k$, en orden:

1. **Predicción previa (LOO secuencial).** Con el modelo vigente, que todavía no tiene $k$, cada persona hereda el reparto espacial de su perfil y se suma por categoría de $k$:
$$
\hat\nu^{(k)}[c,m] = \sum_{i:\,x_{ik}=c} \frac{W[p(i),m]}{\mu_{p(i)}} .
$$
Se compara con el $\nu^{(k)}$ publicado y con el proporcional. El SRMSE se calcula sobre la partición completa de cada manzana, como en C5: categorías publicadas más el *resto* $= n_{per} - \sum$ publicadas. Es la extensión cerrada del nivel 2 aplicada a una variable que *sí* tiene marginal, así que mide directamente qué tan confiable es esa extensión.
2. **Ajuste.** Se agrega $\nu^{(k)}$ como restricción. La semilla es esa misma predicción: el $W$ previo repartido en los perfiles nuevos. Su logaritmo solo tiene términos de las restricciones vigentes, así que, por el argumento de C4, el resultado es el mismo que desde una semilla uniforme, pero converge más rápido. D3c lo verifica.
3. **Chequeos.** Convergencia, error marginal de todos los bloques y, en los bloques con `*`, que las manzanas enmascaradas sumen el residuo comunal.

Esta tanda agrega los cinco bloques con enmascaramiento (nacionalidad, pueblo, afro, lengua, religión), que son binarios y suman pocos perfiles. Notas:
- La predicción previa depende del orden: es un LOO *secuencial*. El LOO completo del modelo final se puede calcular al terminar.
- El SRMSE normaliza por el tamaño medio de celda de la partición, así que en un binario raro (pueblo, lengua) sale chico solo porque la categoría es chica. No se compara entre bloques; lo comparable es la *mejora (%)* respecto del proporcional.
- *Categoría más chica (comuna)* es el total comunal de la categoría publicada más pequeña del bloque: un primer indicador de riesgo para la Fase 2.

In [4]:
# D3a. Regresiones: [edad, sexo, inmigrante] desde el inventario reproduce T_espina, y la predicción de
# inmigrante desde [edad, sexo] reproduce el LOO interno del notebook 02 (C5)
modelo = ajustar_modelo(inv, ["edad", "sexo", "inmigrante"])
T_modelo = np.zeros_like(T_espina)
T_modelo[modelo["perfiles"][:, 0], modelo["perfiles"][:, 1], modelo["perfiles"][:, 2], :] = modelo["W"]
print("perfiles:", len(modelo["mu_p"]), "| iteraciones:", modelo["iteraciones"], "| convergió:", modelo["convergio"])
print("errores marginales:", {b: f"{e:.2e}" for b, e in modelo["errores"].items()})
print("max |T_modelo - T_espina|:", np.abs(T_modelo - T_espina).max())

modelo_es = ajustar_modelo(inv, ["edad", "sexo"])
pred, nu_k, cod = predecir_bloque(inv, modelo_es, "inmigrante")
prop = (np.bincount(cod, minlength=nu_k.shape[0]) / len(cod))[:, None] * nu[None, :]
print("predicción de inmigrante desde [edad, sexo] -> SRMSE proporcional:", round(srmse(prop, nu_k), 4),
      "| predicción:", round(srmse(pred, nu_k), 4))

assert modelo["convergio"] and np.abs(T_modelo - T_espina).max() < 1e-6
assert abs(srmse(prop, nu_k) - esp["loo_inmigrante_proporcional"]) < 5e-4
assert abs(srmse(pred, nu_k) - esp["loo_inmigrante_con_EM"]) < 5e-4

perfiles: 28 | iteraciones: 5242 | convergió: True
errores marginales: {'edad': '2.81e-08', 'sexo': '1.34e-09', 'inmigrante': '4.55e-13', 'filas (mu_p)': '6.58e-07'}
max |T_modelo - T_espina|: 0.0
predicción de inmigrante desde [edad, sexo] -> SRMSE proporcional: 0.9268 | predicción: 0.7813


In [5]:
# D3b. Bloques enmascarados: predicción previa -> ajuste con semilla heredada -> chequeos
orden_d3 = ["nacionalidad", "pueblo", "afro", "lengua", "religion"]
assert modelo["bloques"] == ["edad", "sexo", "inmigrante"], "corre D3a antes"
filas_d3 = []
for bloque in orden_d3:
    pred, nu_k, cod = predecir_bloque(inv, modelo, bloque)
    C = nu_k.shape[0]
    conteo = np.bincount(cod, minlength=C)
    prop = (conteo / len(cod))[:, None] * nu[None, :]

    modelo = ajustar_modelo(inv, modelo["bloques"] + [bloque], previo=modelo)
    err_max = max(modelo["errores"].values())

    # Las manzanas enmascaradas deben absorber el residuo comunal de la categoría publicada
    masc = np.isnan(nu_k[0])
    est_masc = (indicadora(modelo["perfiles"][:, -1], C) @ modelo["W"])[0, masc].sum()
    residuo = conteo[0] - np.nansum(nu_k[0])

    print(f"[{bloque:>12}] perfiles={len(modelo['mu_p']):>4} | it={modelo['iteraciones']:>6} | convergió={modelo['convergio']} "
          f"| max error marginal={err_max:.2e} | {masc.sum()} enmascaradas: estimado={est_masc:.4f} vs residuo={residuo:.0f}")
    assert modelo["convergio"] and err_max < 1e-5 and abs(est_masc - residuo) < 1e-3

    filas_d3.append({"bloque": bloque, "SRMSE proporcional": srmse(prop, nu_k), "SRMSE predicción previa": srmse(pred, nu_k),
                     "mejora (%)": 100 * (1 - srmse(pred, nu_k) / srmse(prop, nu_k)),
                     "categoría más chica (comuna)": int(conteo[:-1].min()),
                     "perfiles": len(modelo["mu_p"]), "perfiles con 1 persona": int((modelo["mu_p"] == 1).sum())})

tabla_d3 = pd.DataFrame(filas_d3).set_index("bloque")
print()
print(tabla_d3.round(4).to_string())
print("\nbloques en el modelo:", modelo["bloques"])

[nacionalidad] perfiles=  56 | it= 12005 | convergió=True | max error marginal=4.71e-07 | 54 enmascaradas: estimado=119.0000 vs residuo=119
[      pueblo] perfiles=  97 | it=  7607 | convergió=True | max error marginal=4.72e-07 | 130 enmascaradas: estimado=247.0000 vs residuo=247
[        afro] perfiles= 162 | it=  8367 | convergió=True | max error marginal=4.51e-07 | 126 enmascaradas: estimado=195.0000 vs residuo=195
[      lengua] perfiles= 257 | it=  7125 | convergió=True | max error marginal=4.53e-07 | 197 enmascaradas: estimado=352.0000 vs residuo=352
[    religion] perfiles= 403 | it= 18925 | convergió=True | max error marginal=1.78e-07 | 4 enmascaradas: estimado=84.0000 vs residuo=84

              SRMSE proporcional  SRMSE predicción previa  mejora (%)  categoría más chica (comuna)  perfiles  perfiles con 1 persona
bloque                                                                                                                               
nacionalidad              0.922

In [6]:
# D3c. La semilla heredada no cambia el resultado: el mismo modelo final desde semilla uniforme
modelo_unif = ajustar_modelo(inv, modelo["bloques"])
dif = np.abs(modelo["W"] - modelo_unif["W"])
print("semilla uniforme -> iteraciones:", modelo_unif["iteraciones"], "| convergió:", modelo_unif["convergio"],
      "| (última adición con semilla heredada:", modelo["iteraciones"], "iteraciones)")
print("max |W(heredada) - W(uniforme)|:", dif.max(), "| en personas totales:", dif.sum())

assert modelo_unif["convergio"] and dif.max() < 1e-4

semilla uniforme -> iteraciones: 23045 | convergió: True | (última adición con semilla heredada: 18925 iteraciones)
max |W(heredada) - W(uniforme)|: 2.4272139853565022e-11 | en personas totales: 5.699736572119193e-09


### D4. Los 27 bloques: mismo IPF en GPU

**Por qué en GPU.** Con los 27 bloques el modelo tiene $|\mathcal P| = 40.946$ perfiles y $W$ tiene 19,2 millones de celdas. Medido en esta máquina, una iteración de IPF toma 1,5 s en CPU y 50 ms en GPU.

**Qué cambia y qué no.**
- *El algoritmo es el mismo.* Las actualizaciones son idénticas a `ipf_perfiles` y van en el mismo orden. La suma por categoría $E_kW$ se hace con `cp.add.at`, porque el entorno no trae cuBLAS ni cuSPARSE. D4a verifica que en GPU se reproduce el modelo de D3.
- *El criterio de parada cambia.* Se para cuando el error marginal máximo, en personas, de todas las restricciones (filas $\mu_p$ incluidas) es $\le$ `tol`, revisado cada 50 iteraciones.
- *Tolerancia.* $10^{-6}$ ya no es alcanzable: con 11 bloques el IPF se estanca en $\sim 4\times10^{-4}$ después de 200.000 iteraciones. Para los 27 bloques usamos `TOL_27 = 0.1` personas. El error que queda se concentra en las manzanas más grandes, donde 0,1 de varios miles de personas es menos de un 0,01%. Además, las tablas finales son conteos enteros y el ruido de la Fase 2 será órdenes de magnitud mayor. Si se quiere más precisión, basta con reanudar desde el $W$ obtenido: es el mismo punto fijo.

**Pasos.**
- D4b predice los 19 bloques nuevos desde el modelo de D3 (predicción previa).
- D4c ajusta una sola vez con los 27 bloques, con semilla heredada, guarda el resultado en `resultados/` (o lo carga si ya existe) y mide cuánto se concentra cada perfil de una persona en una manzana (chequeo de privacidad).
- La tabla bivariada (D4d) y los mapas (D5) están en `04_tablas_y_mapas.ipynb`, que carga el modelo guardado.

In [7]:
# D4a. El mismo IPF en GPU (ipf_perfiles_gpu) reproduce el modelo de D3 ajustado en CPU
bloques_d3 = ["edad", "sexo", "inmigrante"] + orden_d3
assert modelo["bloques"] == bloques_d3, "`modelo` no es el de D3: corre D3a y D3b antes"
modelo_gpu = ajustar_modelo_gpu(inv, bloques_d3)
dif = np.abs(modelo_gpu["W"] - modelo_unif["W"])
print("GPU -> iteraciones:", modelo_gpu["iteraciones"], "| convergió:", modelo_gpu["convergio"],
      f"| {modelo_gpu['segundos']:.1f}s | (CPU en D3c: {modelo_unif['iteraciones']} iteraciones, {modelo_unif['segundos']:.1f}s)")
print("max error marginal:", f"{max(modelo_gpu['errores'].values()):.2e}")
print("max |W(GPU) - W(CPU)|:", f"{dif.max():.2e}", "| en personas totales:", f"{dif.sum():.2e}")

assert modelo_gpu["convergio"] and max(modelo_gpu["errores"].values()) < 1e-5 and dif.max() < 1e-4

GPU -> iteraciones: 20650 | convergió: True | 18.6s | (CPU en D3c: 23045 iteraciones, 50.2s)
max error marginal: 9.65e-07
max |W(GPU) - W(CPU)|: 6.27e-07 | en personas totales: 1.99e-05


In [8]:
# D4b. Predicción previa de los 19 bloques nuevos desde el modelo de D3 (que no tiene ninguno de ellos)
bloques_27 = list(inv.marginales)
nuevos = [b for b in bloques_27 if b not in bloques_d3]
assert modelo["bloques"] == bloques_d3, "`modelo` no es el de D3: corre D3a y D3b antes"

filas_d4 = []
for bloque in nuevos:
    pred, nu_k, cod = predecir_bloque(inv, modelo, bloque)
    conteo = np.bincount(cod, minlength=nu_k.shape[0])
    prop = (conteo / len(cod))[:, None] * nu[None, :]
    filas_d4.append({"bloque": bloque, "categorías": nu_k.shape[0] - 1,
                     "SRMSE proporcional": srmse(prop, nu_k), "SRMSE predicción previa": srmse(pred, nu_k),
                     "mejora (%)": 100 * (1 - srmse(pred, nu_k) / srmse(prop, nu_k)),
                     "categoría más chica (comuna)": int(conteo[:-1].min())})
tabla_d4 = pd.DataFrame(filas_d4).set_index("bloque")
print(len(nuevos), "bloques nuevos")
print(tabla_d4.round(4).to_string())

19 bloques nuevos
                categorías  SRMSE proporcional  SRMSE predicción previa  mejora (%)  categoría más chica (comuna)
bloque                                                                                                           
dif_ver                  1              0.0691                   0.0504     27.1220                          4197
dif_oir                  1              0.0410                   0.0204     50.3116                          1776
dif_mover                1              0.1027                   0.0504     50.9430                          4298
dif_cogni                1              0.0594                   0.0449     24.4094                          2486
dif_cuidado              1              0.0672                   0.0545     18.9788                          1694
dif_comunic              1              0.0328                   0.0284     13.5753                          1033
discapacidad             1              0.1607                   0.081

In [9]:
# D4c. Los 27 bloques: un ajuste en GPU con semilla heredada del modelo de D3 (o se carga si ya está guardado)
TOL_27 = 0.1       # personas (ver markdown de D4)
REAJUSTAR = False  # True fuerza el ajuste aunque exista el archivo (~15-20 min en GPU)
ruta_27 = os.path.join("resultados", f"modelo_27_{COMUNA_CUT}.npz")
ruta_d3 = os.path.join("resultados", f"modelo_d3_{COMUNA_CUT}.npz")

if os.path.exists(ruta_27) and not REAJUSTAR:
    modelo_27 = cargar_modelo(ruta_27)
    print("cargado desde", ruta_27)
else:
    modelo_27 = ajustar_modelo_gpu(inv, bloques_27, previo=modelo, tol=TOL_27, n_iter=100000, reportar=2500)
    guardar_modelo(modelo_27, ruta_27)
    print("guardado en", ruta_27)
guardar_modelo(modelo, ruta_d3)  # el modelo de D3 lo usa el notebook 04 para las predicciones previas

# El modelo (ajustado o cargado) debe corresponder a los perfiles del inventario actual
_, perfiles_27, _, mu_p_27 = construir_perfiles({b: codigos_bloque(inv, b)[0] for b in bloques_27})
assert modelo_27["bloques"] == bloques_27
assert np.array_equal(perfiles_27, modelo_27["perfiles"]) and np.array_equal(mu_p_27, modelo_27["mu_p"])

errores_27 = errores_modelo(inv, modelo_27)
print(f"\nperfiles={len(modelo_27['mu_p'])} | con 1 persona={int((modelo_27['mu_p'] == 1).sum())} "
      f"| celdas W={modelo_27['W'].size:,} | iteraciones={modelo_27['iteraciones']} "
      f"| convergió={modelo_27['convergio']} | ajuste: {modelo_27['segundos'] / 60:.1f} min")
for b, e in errores_27.items():
    print(f"  error marginal {b:<15}: {e:.2e}")

# Las manzanas enmascaradas absorben el residuo comunal (a la tolerancia del ajuste)
for j, b in enumerate(bloques_27):
    cod, nu_k = codigos_bloque(inv, b)
    masc = np.isnan(nu_k[0])
    if masc.any():
        est = (indicadora(modelo_27["perfiles"][:, j], nu_k.shape[0]) @ modelo_27["W"])[0, masc].sum()
        residuo = (cod == 0).sum() - np.nansum(nu_k[0])
        print(f"  {b:<13} enmascaradas={masc.sum():>3}: estimado={est:.3f} vs residuo={residuo:.0f}")

# Privacidad: cuánto se concentra en una sola manzana cada perfil de una persona
reparto = modelo_27["W"][modelo_27["mu_p"] == 1]
max_reparto = reparto.max(axis=1)
n_efectivo = 1 / (reparto ** 2).sum(axis=1)
print(f"\nperfiles de 1 persona: reparto máximo mediano={np.median(max_reparto):.3f} | con >50% en una manzana: "
      f"{(max_reparto > 0.5).mean():.2%} | con >90%: {(max_reparto > 0.9).mean():.2%} "
      f"| n° efectivo de manzanas (mediana)={np.median(n_efectivo):.1f}")

assert modelo_27["convergio"], "no llegó a TOL_27: reanudar con previo=modelo_27"
assert max(errores_27.values()) <= TOL_27

    it=  2500 | error marginal=7.17e-01 | 129s
    it=  5000 | error marginal=4.64e-01 | 258s
    it=  7500 | error marginal=2.78e-01 | 388s
    it= 10000 | error marginal=1.84e-01 | 519s
    it= 12500 | error marginal=1.37e-01 | 649s
    it= 15000 | error marginal=1.09e-01 | 778s
guardado en resultados/modelo_27_13108.npz

perfiles=40946 | con 1 persona=30742 | celdas W=19,203,674 | iteraciones=16100 | convergió=True | ajuste: 13.9 min
  error marginal edad           : 3.92e-03
  error marginal sexo           : 4.19e-04
  error marginal inmigrante     : 1.80e-03
  error marginal nacionalidad   : 1.78e-03
  error marginal pueblo         : 6.86e-05
  error marginal afro           : 5.80e-05
  error marginal lengua         : 5.09e-05
  error marginal religion       : 2.84e-03
  error marginal dif_ver        : 4.34e-04
  error marginal dif_oir        : 1.07e-04
  error marginal dif_mover      : 4.57e-04
  error marginal dif_cogni      : 2.55e-04
  error marginal dif_cuidado    : 3.26e-04
